# VORTEX-HRM Benchmark on Google Colab

Runs the 50-QA benchmark with automatic checkpoint resume.

**How it works:**
1. Mount Google Drive — checkpoints saved to `MyDrive/vortex-hrm-benchmark/`
2. Clone repo (public, no token)
3. Install ollama + pull `qwen2.5:7b`
4. Run `batch_runner.py` — it saves checkpoint **after every question**
5. If session drops: re-run → mounts Drive → detects existing checkpoints → resumes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/dizel0110/Text-HRM-RAG/blob/main/notebooks/vortex_benchmark_colab.ipynb)

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

DRIVE_DIR = "/content/drive/MyDrive/vortex-hrm-benchmark"
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Checkpoints → {DRIVE_DIR}")

### 2. Clone repository

In [ ]:
REPO_DIR = "/content/Text-HRM-RAG"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/dizel0110/Text-HRM-RAG.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}/vortex-hrm
!pip install -r requirements.txt -q
print("Repo ready")

### 3. Install ollama + pull model

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
print("ollama + qwen2.5:7b ready")

### 4. Show checkpoint status

In [ ]:
checkpoint_file = os.path.join(DRIVE_DIR, "checkpoint.json")
if os.path.exists(checkpoint_file):
    import json
    with open(checkpoint_file) as f:
        done = json.load(f)
    print(f"Resuming: {len(done)}/50 already completed")
else:
    print("Fresh run: 0/50 completed")

### 5. Run benchmark

Uses `configs/local.yaml` (ollama + qwen2.5:7b) with output to Google Drive.

In [ ]:
!python scripts/batch_runner.py \
    --config configs/local.yaml \
    --questions data/multi_domain/questions.json \
    --corpus data/multi_domain/corpus.json \
    --output {DRIVE_DIR}
print("\nDone.")

### 6. Evaluate results

In [ ]:
!python scripts/eval.py --predictions {DRIVE_DIR}/predictions.jsonl
print("\nFull results in Drive: MyDrive > vortex-hrm-benchmark")

### 7. (Optional) Download results locally

In [ ]:
from google.colab import files
files.download(f"{DRIVE_DIR}/predictions.jsonl")